# Train GPT-BERT on native (non-translated) Hindi/Telugu data

Trains monolingual GPT-BERT models from scratch on the CC-100-derived native data (`pulipakav-1/hi-te`), using this project's existing `babybabellm-gptbert` training code (`Babylm2026/gpt-bert/{hindi,telugu}`), scaled down to fit a single Colab GPU in a few hours instead of the original multi-GPU cluster-scale run (`max_steps=15625`, `global_batch_size=32768` across 3-4 GPUs).

**Trains Hindi then Telugu, one after the other, in a single pass** -- just run all cells top to bottom.

Steps per language: download native data -> train a fresh tokenizer on it -> shard/tokenize the data -> train the model.

In [ ]:
# Cell 1: clone the repo once and install dependencies for both languages
!git clone https://github.com/vishnup22/BabyLM.git
%cd BabyLM
!git checkout evaluation
!git pull
!pip install -q -r Babylm2026/gpt-bert/hindi/requirements.txt
!pip install -q -r Babylm2026/gpt-bert/telugu/requirements.txt

import os
REPO_ROOT = os.getcwd()

In [ ]:
# Cell 2: log in to Hugging Face (only needed if you want the optional push-to-HF step at
# the end -- pulipakav-1/hi-te itself is a public dataset, no token needed just to read it)
from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")  # use whatever secret name you saved your token under
!hf auth whoami

In [ ]:
# Cell 3: hyperparameters -- SCALED DOWN from the original cluster-scale run
# (max_steps=15625, global_batch_size=32768 across 3-4 GPUs) to fit a single Colab GPU in
# a few hours. Watch the first ~50 steps' pace (printed by the training loop) and adjust
# MAX_STEPS if you want a longer or shorter run. Model architecture is unchanged
# (configs/base.json).
MAX_STEPS = 2000
GLOBAL_BATCH_SIZE = 256
LOCAL_BATCH_SIZE = 64  # lower this if you hit an out-of-memory error on a smaller GPU
SEQ_LENGTH = 128

In [ ]:
# Cell 4: train both languages, one after the other
from pathlib import Path
from huggingface_hub import hf_hub_download

for LANG in ["hindi", "telugu"]:
    LANG_CODE = {"hindi": "hi", "telugu": "te"}[LANG]
    DATASET_NAME = f"native-{LANG}"

    print(f"\n{'=' * 70}\nTraining {LANG}\n{'=' * 70}")
    %cd {REPO_ROOT}/Babylm2026/gpt-bert/{LANG}

    # download the native data and lay it out in the structure the tokenizer/shard
    # tools expect: data/raw/<dataset_name>/<dataset_name>.train.<lang>.txt
    src_path = hf_hub_download(repo_id="pulipakav-1/hi-te", filename=f"{LANG}.txt", repo_type="dataset")
    raw_dir = Path("data/raw") / DATASET_NAME
    raw_dir.mkdir(parents=True, exist_ok=True)
    dest_path = raw_dir / f"{DATASET_NAME}.train.{LANG_CODE}.txt"
    dest_path.write_bytes(Path(src_path).read_bytes())
    print(f"Placed {dest_path} ({dest_path.stat().st_size:,} bytes)")

    # train a fresh tokenizer on the native data (vocab_size=16384, matching the
    # monolingual config -- reusing the old translated-data tokenizer here would
    # reintroduce the tokenizer-granularity issue found earlier in this project)
    !python tools/train_tokenizer_local.py \
      --dataset {DATASET_NAME} \
      --data_root data/raw \
      --output tokenizers/tokenizer_native_16384.json \
      --vocab_size 16384

    # tokenize the data and write train/valid shards (2% held out for validation)
    !python tools/prepare_local_shards.py \
      --dataset {DATASET_NAME} \
      --data_root data/raw \
      --tokenizer tokenizers/tokenizer_native_16384.json \
      --output_base data/processed \
      --valid_fraction 0.02

    # train, single GPU
    os.environ["WANDB_MODE"] = "disabled"  # no W&B account needed
    %cd pretraining
    !python train_single_gpu.py \
      --train_path ../data/processed/train \
      --valid_path ../data/processed/valid \
      --config_file ../configs/base.json \
      --tokenizer_path ../tokenizers/tokenizer_native_16384.json \
      --name native-{LANG}-gptbert \
      --output_dir ../model_checkpoints \
      --hybrid_numerator 2 \
      --hybrid_denominator 3 \
      --global_batch_size {GLOBAL_BATCH_SIZE} \
      --local_batch_size {LOCAL_BATCH_SIZE} \
      --seq_length {SEQ_LENGTH} \
      --max_steps {MAX_STEPS} \
      --save_every 200 \
      --validate_every 0 \
      --seed 42

    print(f"\nFinished training {LANG}.")

## Optional: push both trained checkpoints to Hugging Face

Requires the HF login cell above to have been run with a write-scoped token.

In [ ]:
# Cell 5 (optional): push the checkpoint + tokenizer + config for both languages
from huggingface_hub import HfApi

api = HfApi()
for LANG in ["hindi", "telugu"]:
    lang_dir = f"{REPO_ROOT}/Babylm2026/gpt-bert/{LANG}"
    repo_id = f"pulipakav-1/native-{LANG}-gptbert"
    api.create_repo(repo_id=repo_id, repo_type="model", exist_ok=True)
    for local_path, repo_path in [
        (f"{lang_dir}/model_checkpoints/native-{LANG}-gptbert_2_3_ema.bin", "model_ema.bin"),
        (f"{lang_dir}/tokenizers/tokenizer_native_16384.json", "tokenizer.json"),
        (f"{lang_dir}/configs/base.json", "config_base.json"),
    ]:
        api.upload_file(path_or_fileobj=local_path, path_in_repo=repo_path, repo_id=repo_id, repo_type="model")
    print(f"Pushed to https://huggingface.co/{repo_id}")